In [1]:
import pandas as pd

import warnings
warnings.filterwarnings('ignore')


In [168]:
yerevan_historical = pd.read_csv('../database/yerevan_historical_clean/sale_historical.csv')
yerevan_actual = pd.read_csv('../database/yerevan_actual_clean/sale_actual.csv')

## Modifying historical data columns

In [169]:
yerevan_historical.drop(columns=[ 'days_in'], inplace=True)

In [170]:
yerevan_actual['backup_status'] = 'not sold'

In [171]:
df = pd.concat([yerevan_historical, yerevan_actual], axis=0)

In [172]:
df.drop(columns=['links', 'source', 'rent_or_sale'], inplace=True)

In [173]:
df[df['id']==10895]

,backup_status,id,price,rooms,square_meters,address,sold_date,furniture,renovation,price_per_meter,...,height,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered
33,sold,10895,100000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",2022-10-08 21:56:34,1.0,NaN,1851.851852,...,2.8,1.0,2022-08-28 19:49:22,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513106,40.166187
17563,not sold,10895,125000.0,3.0,54.0,"Yerevan, Center, Tigran Mets Ave",NaN,1.0,NaN,2314.814815,...,2.8,1.0,2023-03-13 18:12:05,44.513112,40.166182,Zoravar Andranik,669,Kentron,44.513104,40.166181


In [174]:
df = df.drop_duplicates(subset=['id'], keep='last')
df = df.loc[~((df['backup_status'] == 'sold') & (df['id'].duplicated()))]

In [175]:
df = df.dropna(subset=['district']).reset_index(drop=True)

In [176]:
df_test = pd.read_csv('../database/yerevan_combined_clean/sale_combined.csv')
df_test

,backup_status,id,price,rooms,square_meters,address,furniture,renovation,price_per_meter,floor,...,bathroom,download_date,y,x,closest_metro,walking_distance_to_metro(m),district,latitude_jittered,longitude_jittered,neighborhood
0,sold,1055,130000.0,2.0,55.0,"Yerevan, Center, Vardanants St",1.0,NaN,2054.545455,3.0,...,1.0,2022-09-10 02:04:46,44.521089,40.174314,Republic Square,921,Kentron,44.521096,40.174314,Kentron
1,sold,1991,95000.0,2.0,90.0,"Yerevan, Arabkir, V.Papazyan St",NaN,NaN,1800.000000,4.0,...,1.0,NaN,44.502252,40.204200,Barekamutyun,1196,Arabkir,44.502244,40.204205,Nor_Arabkir
2,sold,4351,80000.0,2.0,42.0,"Yerevan, Arabkir,",NaN,NaN,1642.833333,3.0,...,1.0,NaN,44.509420,40.207286,Barekamutyun,1903,Arabkir,44.509412,40.207290,Nor_Arabkir
3,sold,4420,110000.0,3.0,50.0,"Yerevan, Arabkir, Aghbyur Serob St",NaN,NaN,1800.000000,NaN,...,1.0,NaN,44.503042,40.196823,Baghramyan,694,Arabkir,44.503041,40.196827,Nor_Arabkir
4,sold,4862,65000.0,2.0,90.0,"Yerevan, Davtashen, Davtashen 1 district",NaN,NaN,1555.555556,11.0,...,1.0,NaN,44.488840,40.220763,Barekamutyun,3271,Davtashen,44.488848,40.220768,1st_Block
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73974,not sold,113965,190000.0,3.0,79.0,Էրեբունու,0.0,NaN,2405.063291,4.0,...,NaN,2023-03-13 18:55:07,44.522257,40.141947,Sasuntsi Davit,2956,Erebuni,44.522248,40.141940,Nor_Aresh
73975,not sold,87453,300000.0,3.0,114.0,Ծիծեռնակաբերդի խճ.,0.0,NaN,2631.578947,13.0,...,NaN,2023-03-13 18:55:14,44.485534,40.185835,Barekamutyun,2596,Kentron,44.485543,40.185844,Kentron
73976,not sold,102196,185000.0,3.0,96.0,Կոմիտասի պող.,0.0,NaN,1927.083333,3.0,...,NaN,2023-03-13 18:56:07,44.507210,40.206239,Barekamutyun,1677,Arabkir,44.507205,40.206245,Nor_Arabkir
73977,not sold,106857,98000.0,2.0,48.0,"Կոմիտասի պող. 23/1, բն. 10",0.0,NaN,2041.666667,4.0,...,NaN,2023-03-13 18:56:08,44.502342,40.205426,Barekamutyun,1468,Arabkir,44.502340,40.205432,Nor_Arabkir


In [177]:
df_7 = pd.merge(df, df_test[['id', 'neighborhood']], on='id', how='left')

In [178]:
df['neighborhood'] = df_7['neighborhood']

### classifying neighborhoods

In [55]:
df2 = df.head(75732).reset_index(drop=True)

In [32]:
import requests

api_key = "***REMOVED-GOOGLE-API-KEY***"

addresses = []

In [33]:
while True:
    try:
        for index, row in df2.iterrows():
            latlng = f"{row['x']},{row['y']}"
            url = f"https://maps.googleapis.com/maps/api/geocode/json?latlng={latlng}&location_type=APPROXIMATE&key={api_key}"
            response = requests.get(url)
            data = response.json()
            address = data['results'][0]['address_components'][0]['long_name']
            # print(address)
            df2.drop(index=df2.index[0], axis=0, inplace=True)
            addresses.append(address)
    except:
        # if there was an error, print a message and try again
        print("An error occurred. Retrying...")
        continue
    else:
        # if there was no error, break out of the loop and continue with the rest of the code
        break

An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error occurred. Retrying...
An error

In [87]:
df['neighborhood'] = addresses

In [ ]:
df.loc[df['neighborhood'] == '0054', 'neighborhood'] = 'Davtashen Block'
df.loc[df['neighborhood'] == '0059', 'neighborhood'] = 'Nor Nork 9th Microdistrict'

df['neighborhood'] = df['neighborhood'].str.replace(' ', '_')
df['neighborhood'] = df['neighborhood'].str.replace('-', '_')

In [87]:
df.to_csv(r'../database/yerevan_combined_clean/sale_combined_neighborhood.csv', index=False)

### small amount of rows weren't classified

In [88]:
df = pd.read_csv('../database/yerevan_combined_clean/sale_combined_neighborhood.csv')

In [182]:
df = df.dropna(subset=['neighborhood'])

In [184]:
sale_historical = df[df['backup_status']=='sold'].reset_index(drop=True)
sale_actual = df[df['backup_status']=='not sold'].reset_index(drop=True)

In [186]:
# removing neighborhoods with low counts
neighborhood_counts = sale_historical['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_historical = sale_historical[~sale_historical['neighborhood'].isin(neighborhoods_to_remove)]

In [187]:
# removing neighborhoods with low counts
neighborhood_counts = sale_actual['neighborhood'].value_counts()
neighborhood_df = pd.DataFrame({'neighborhood': neighborhood_counts.index, 'count': neighborhood_counts.values})
neighborhoods_to_remove = neighborhood_df[neighborhood_df['count'] < 5]['neighborhood'].tolist()
sale_actual = sale_actual[~sale_actual['neighborhood'].isin(neighborhoods_to_remove)]

In [196]:
sale_actual.drop(columns=['sold_date'], inplace=True)

In [199]:
sale_historical['sold_date'] = pd.to_datetime(sale_historical['sold_date'])

# Extract the date part from the datetime column
sale_historical['sold_date'] = sale_historical['sold_date'].dt.date

In [200]:
sale_historical.to_csv(r'../database/yerevan_combined_clean/historical_sold.csv', index=False)
sale_actual.to_csv(r'../database/yerevan_combined_clean/actual_not_sold.csv', index=False)

In [201]:
df = pd.concat([sale_historical,sale_actual], axis=0)

In [202]:
df.drop(columns=[ 'sold_date'], inplace=True)

In [203]:
df_clean = df.dropna().reset_index(drop=True)

In [204]:
df.to_csv(r'../database/yerevan_combined_clean/sale_combined.csv', index=False)
df_clean.to_csv(r'../database/yerevan_combined_clean/sale_clean_combined.csv', index=False)